### Phase 2: Feature Engineering

### Step 1: Load cleaned data

In [2]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv('../data/cleaned/orders_cleaned.csv')

In [4]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

### Step 2: Re-convert datetime columns

CSV files don't preserve pandas data types — when a DataFrame is saved to 
CSV and reloaded, datetime columns come back as plain text (`object`). 
Even though these columns were already converted in `02_data_cleaning.ipynb`, 
that conversion doesn't survive the save/reload cycle, so it must be 
repeated here before any date-based calculations can be done.

In [5]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [6]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


### Step 3: Calculate delivery time

`delivery_time_days` = number of days between when the customer placed the 
order and when it was actually delivered. This is a core metric for Q3 — 
it lets us measure and compare delivery speed across regions and categories, 
and check its relationship with review scores.

Note: this will be null for any order that was never delivered (matches the 
`order_delivered_customer_date` nulls investigated in the cleaning notebook) 
— that's expected and correct, not an error.

In [7]:
orders['delivery_time_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

In [8]:
orders[['order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_time_days']].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


### Step 4: Calculate delivery delay

`delivery_delay_days` = actual delivery date minus estimated delivery date.

- Negative value → delivered early
- Positive value → delivered late
- 0 → delivered exactly on the estimated date

This is the key metric for Q3 — it measures Olist's ability to meet its own 
delivery promise, which is a more meaningful satisfaction driver than raw 
delivery time alone (a 15-day delivery that was estimated at 20 days is a 
good outcome; an 8-day delivery estimated at 5 days is a bad one).

In [9]:
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

In [10]:
orders[['order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days']].head()

,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,2017-10-10 21:25:13,2017-10-18,-8.0
1,2018-08-07 15:27:45,2018-08-13,-6.0
2,2018-08-17 18:06:29,2018-09-04,-18.0
3,2017-12-02 00:28:42,2017-12-15,-13.0
4,2018-02-16 18:17:02,2018-02-26,-10.0


### Step 5: Check delivery delay distribution

Confirming the delay pattern holds across the full dataset, not just the 
first 5 rows — checking how many orders were early, on-time, or late.

In [11]:
print(f"Early deliveries : {(orders['delivery_delay_days'] < 0).sum()}")
print(f"On-time deliveries : {(orders['delivery_delay_days'] == 0).sum()}")
print(f"Late deliveries : {(orders['delivery_delay_days'] > 0).sum()}")

Early deliveries : 88649
On-time deliveries : 1292
Late deliveries : 6535


### Step 6: Calculate approval time

`approval_time_hours` = time between when the order was placed and when it 
was approved (e.g., payment confirmed). Measured in hours rather than days, 
since approval is usually a fast operational step — a smaller, earlier-stage 
metric than delivery time, useful for spotting bottlenecks right at the 
start of the order lifecycle.

In [12]:
orders['approval_time_hours'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600

In [13]:
orders[['order_purchase_timestamp', 'order_approved_at', 'approval_time_hours']].head()

,order_purchase_timestamp,order_approved_at,approval_time_hours
0,2017-10-02 10:56:33,2017-10-02 11:07:15,0.178333
1,2018-07-24 20:41:37,2018-07-26 03:24:27,30.713889
2,2018-08-08 08:38:49,2018-08-08 08:55:23,0.276111
3,2017-11-18 19:28:06,2017-11-18 19:45:59,0.298056
4,2018-02-13 21:18:39,2018-02-13 22:20:29,1.030556


In [14]:
orders.to_csv('../data/cleaned/orders_cleaned.csv', index=False)